# Three-Regime Benchmark — LLPSense Dataset

Evaluates the LLPSense feature basis (1024 ProtT5 dims + 13 environmental parameters)
with their published Optuna-tuned XGBoost hyperparameters, under four splits that ask
four different scientific questions.

| Regime | Held out | Question |
|---|---|---|
| **A** | whole experimental records | new conditions for a protein already characterised |
| **B** | whole sequences | a new protein |
| **C** | their `cluster/group` | a new protein family, as LLPSense splits it |
| **C\*** | homology-hardened groups | a new protein family, with homology leakage removed |

C vs C\* is our own contribution: 26 of 207 homology clusters are split across more
than one of their groups, so their `GroupKFold` lets homologues sit on both sides of
a fold. C\* closes that.

**Setup:** Runtime -> Change runtime type -> **T4 GPU**. About 10-15 min on GPU.

**Upload when prompted:** `llpsense_final_noseq.csv` and `llpsense_t5.npz`.

## 1. Setup

In [ ]:
import xgboost, sklearn, numpy as np, pandas as pd
print("xgboost", xgboost.__version__, "| sklearn", sklearn.__version__)
import subprocess
GPU = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
print("GPU available:", GPU, "(without one this takes ~80 min instead of ~12)")

## 2. Upload the data

In [ ]:
from google.colab import files
up = files.upload()        # llpsense_final_noseq.csv  AND  llpsense_t5.npz

df  = pd.read_csv("llpsense_final_noseq.csv")
z   = np.load("llpsense_t5.npz", allow_pickle=True)
emb = pd.DataFrame(z["X"], index=z["SeqID"])
print("rows", len(df), "| embeddings", emb.shape)
print("records", df.RecordID.nunique(), "| sequences", df.SeqID.nunique(),
      "| their groups", df.Group.nunique(), "| hardened groups", df.GroupX.nunique())

## 3. Build the feature matrix

`w_record = 1 / (rows for that experiment)`. Expansion is label-biased — negative
records expand to 20.5 rows on average, positives to 13.0, which drags the row-level
positive rate from 54.8% down to 43.5%. Weighting restores one-experiment-one-vote,
so every metric below is record-weighted.

In [ ]:
COND = ['Conc','pH','temp','MgCl2','NaCl','KCl','PEG300-1k','PEG3k-6k','PEG8k-20k',
        'Ficoll','Dextran -40','Dextran 70-','Glycerol']
X = np.hstack([emb.loc[df.SeqID].to_numpy(np.float32), df[COND].to_numpy(np.float32)])
y = df.Label.to_numpy()
w = df.w_record.to_numpy()
print("feature matrix:", X.shape, "(1024 ProtT5 + 13 conditions)")
print(f"positive rate — rows {y.mean():.3f} | record-weighted {np.average(y, weights=w):.3f}")

## 4. Their published hyperparameters

Copied verbatim from `configs/LLPSense.yaml`, so this is their model, not a retuned one.

In [ ]:
HP = dict(n_estimators=877, max_depth=14, learning_rate=0.023722174554030036,
          min_child_weight=2, colsample_bytree=0.5655509178160505,
          subsample=0.5482175662874387, gamma=0.0012682360748828933,
          reg_lambda=0.3024802937837069, reg_alpha=0.04238171400820899,
          random_state=42)
RUNTIME = dict(tree_method="hist", eval_metric="logloss",
               **({"device": "cuda"} if GPU else {"n_jobs": -1}))
print(HP); print(RUNTIME)

## 5. Run the benchmark

In [ ]:
import time, warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             f1_score, brier_score_loss)
from xgboost import XGBClassifier

REGIMES = [("A  new condition, known protein", "RecordID"),
           ("B  new protein",                  "SeqID"),
           ("C  new family (their groups)",    "Group"),
           ("C* new family (hardened)",        "GroupX")]

results = {}
for name, key in REGIMES:
    g, rows, t0 = df[key].to_numpy(), [], time.time()
    for k, (i, j) in enumerate(StratifiedGroupKFold(5, shuffle=True, random_state=1)
                               .split(X, y, groups=g)):
        m = XGBClassifier(**HP, **RUNTIME)
        m.fit(X[i], y[i], sample_weight=w[i])
        p = m.predict_proba(X[j])[:, 1]
        if len(np.unique(y[j])) < 2:
            continue
        rows.append([roc_auc_score(y[j], p, sample_weight=w[j]),
                     average_precision_score(y[j], p, sample_weight=w[j]),
                     accuracy_score(y[j], p > .5, sample_weight=w[j]),
                     f1_score(y[j], p > .5, sample_weight=w[j]),
                     brier_score_loss(y[j], p, sample_weight=w[j])])
    results[name] = np.array(rows)
    a = results[name]
    print(f"{name:34s} ROC-AUC {a[:,0].mean():.3f} ±{a[:,0].std():.3f}   ({time.time()-t0:.0f}s)")

## 6. Results table

In [ ]:
tab = pd.DataFrame({
    n: {"ROC-AUC": f"{a[:,0].mean():.3f} ± {a[:,0].std():.3f}",
        "PR-AUC":  f"{a[:,1].mean():.3f}",
        "Accuracy":f"{a[:,2].mean():.3f}",
        "F1":      f"{a[:,3].mean():.3f}",
        "Brier":   f"{a[:,4].mean():.3f}"} for n, a in results.items()}).T
base = np.average(y, weights=w)
print(tab.to_string())
print(f"\nmajority baseline: ROC-AUC 0.500 | PR-AUC {base:.3f} | accuracy {max(base,1-base):.3f}")

cost = results["C  new family (their groups)"][:,0].mean() - results["C* new family (hardened)"][:,0].mean()
print(f"\ncost of their homology leak (C - C*): {cost:+.3f} ROC-AUC")
gap  = results["A  new condition, known protein"][:,0].mean() - results["B  new protein"][:,0].mean()
print(f"condition-screening vs novel-protein gap (A - B): {gap:+.3f} ROC-AUC")

tab.to_csv("regime_benchmark_results.csv")
np.save("regime_folds.npy", results, allow_pickle=True)
from google.colab import files
files.download("regime_benchmark_results.csv")

### How to read this

- **A high, B and C low** reproduces on their data what we found on yours: these models
  screen conditions well for known proteins and transfer poorly to new ones.
- **C\* below C** is the price of their homology leak, quantified for the first time.
- Send `regime_benchmark_results.csv` back and I'll fold it into the write-up.